In [2]:
# Run in conda ibiz environment
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import pyproj
from scipy.spatial import cKDTree

# importlib seems to clutter the kernel

# CONFIGS
from preprocess_radargrams_configs import PATH_2016_AN_UTIG_ER2HI1B, PATH_2016_AN_UTIG_ER2HI2, CODE_1B, CODE_2
from preprocess_radargrams_configs import MATCHING_TOLERANCE, TX_OFFSET_HiCARS1_microseconds, C, EPSILON_ICE
from preprocess_radargrams_configs import AMPLITUDE_VMIN, AMPLITUDE_VMAX, COLOR_MAP
from preprocess_radargrams_configs import IMAGE_TILE_WIDTH, IMAGE_TILE_HEIGHT

from paths import path_to_ER2HI1B_2016, path_to_ER2HI2_2016_ice_thickness

In [3]:
# Select one radargram
example_radargram_name = "ER2HI1B_2017024_TOT1_JKB2r_X19a_000.nc"
# The ice thickness files usually covers multiple granules 
example_icethickness_name = "ER2HI2_2017024_TOT1_JKB2r_X19a_icethk.txt"

# Define the path to the example radargram .nc file
path_to_example_radargram = os.path.join(path_to_ER2HI1B_2016, example_radargram_name)
path_to_example_ice_thickness = os.path.join(path_to_ER2HI2_2016_ice_thickness, example_icethickness_name)

# Load radargram

In [4]:
radargram_ds = xr.open_dataset(path_to_example_radargram)
print("Granule ID:", radargram_ds.granule_id)
print("Radargram shape:", radargram_ds["amplitude_high_gain"].shape)

radargram_ds

Granule ID: ER2HI1B_2017024_TOT1_JKB2r_X19a_000
Radargram shape: (4000, 3200)


<xarray.Dataset> Size: 103MB
Dimensions:              (fasttime_dim: 3200, time: 4000)
Coordinates:
  * time                 (time) datetime64[ns] 32kB 2017-01-24T07:55:32.56640...
Dimensions without coordinates: fasttime_dim
Data variables:
    fasttime             (fasttime_dim) float32 13kB ...
    lat                  (time) float32 16kB ...
    lon                  (time) float32 16kB ...
    altitude             (time) float32 16kB ...
    pitch                (time) float32 16kB ...
    roll                 (time) float32 16kB ...
    heading              (time) float32 16kB ...
    amplitude_low_gain   (time, fasttime_dim) float32 51MB ...
    amplitude_high_gain  (time, fasttime_dim) float32 51MB ...
Attributes: (12/27)
    featureType:          timeSeries
    location:             C-GJKB
    source:               airborne ice penetrating radar
    data_version:         JUN292018
    netcdf_version:       4.1.2 of Apr 12 2011 15:15:12 $
    positioning:          transect based Ashtech CA GG24 reciever (no orrieta...
    ...                   ...
    rfparams:             RF PARAMETERS\nCenter Frequency: 60 MHz\nCenter Wav...
    digital:              DIGITAL PARAMETERS\nRecord duration: 64 microsecond...
    TX-record_offset:     Transmit delay from record start: 2.550 microseconds\n
    antenna:              ANTENNA PARAMETERS:\nPlatform: C-GJKB DC-3T aircraf...
    processing:           PROCESSING PARAMETERS\nMode name: pik1\nGoal: emula...
    references:           Peters, M.E., D.D. Blankenship, S.P. Carter, D.A. Y...

In [5]:
print("Flight:", radargram_ds.attrs["flight"])
print("TX-record_offset:", radargram_ds.attrs["TX-record_offset"])
# This is very important for processing correctly. 
print(radargram_ds.attrs["processing"].splitlines()[-1])
print("Latitude values:")
print(radargram_ds.lat.values[0:3])

Flight: ICP8/F08
TX-record_offset: Transmit delay from record start: 2.550 microseconds

Typical trace spacing: 22.5 meters
Latitude values:
[-67.6101  -67.61002 -67.60995]


## The transmit is delayed 

In [6]:
print(TX_OFFSET_HiCARS1_microseconds)

with np.printoptions(suppress = True, precision = 3):
    print(radargram_ds.fasttime.values[127:])

2.55
[ 2.54  2.56  2.58 ... 63.94 63.96 63.98]


In [7]:
# Make a new coordinate variable for two-way travel time (TWTT) in microseconds, adjusted for the TX-record offset
radargram_ds = radargram_ds.assign_coords(
    twtt_us = ("fasttime_dim", radargram_ds.fasttime.values - TX_OFFSET_HiCARS1_microseconds)
)

radargram_ds["twtt_us"].attrs["units"] = "microseconds"
radargram_ds["twtt_us"].attrs["long_name"] = "two-way travel time since pulse transmission"

print(radargram_ds["twtt_us"].values[:3], radargram_ds["twtt_us"].values[125:130])

[-2.55 -2.53 -2.51] [-0.04999995 -0.02999997 -0.00999999  0.00999999  0.02999997]


# Load ice thickness

lon and lat in the text file are reported to 6 decimals

In [8]:
# Load header lines (starting with #) from the L2 file first
with open(path_to_example_ice_thickness, "r") as f:
    header_lines = [line for line in f if line.startswith("#")]

# grab last header line for the column headers
col_names = header_lines[-1].lstrip("#").split()

# now import the data into a pandas dataframe, skipping the header lines, but adding the column names from the last header line
l2_df_raw = pd.read_csv(
    path_to_example_ice_thickness,
    sep = r"\s+", 
    # skips any line starting with '#'
    comment = "#", 
    header = None, 
    names = col_names,
)

print("L2 shape (rows):", l2_df_raw.shape[0])

# SANITY CHECK: Completely remove rows with NaN values in the LON or LAT as they are not useful for our purposes
# print(f"L2 dataframe shape before dropping rows with NaN in LON or LAT: {l2_df_raw.shape[0]}")
# Drop them so the the lookup will just not return a match for those rows, instead of returning a match with NaN values.
l2_df = l2_df_raw.dropna(subset = ["LON", "LAT"]) 
# print(f"L2 dataframe shape after dropping rows with NaN in LON or LAT: {l2_df.shape[0]}")
print(f"L2 number of columns missing LON, LAT or BOTH: {l2_df_raw.shape[0] - l2_df.shape[0]}")

# if the L2 dataframe has no valid rows after dropping NaN values, skip this granule
if l2_df.shape[0] == 0:
    print(f"WARNING: L2 dataframe {path_to_example_ice_thickness} has no valid rows after dropping NaN in LON or LAT")

L2 shape (rows): 6408
L2 number of columns missing LON, LAT or BOTH: 0


In [9]:
l2_df_raw

,YEAR,DOY,SOD,LON,LAT,THK,SRF_RNG,BED_ELEVATION,SURFACE_ELEVATION,PARTIAL_BED_REFLECT,SRF_REFLECT,AIRCRAFT_ROLL
0,2017,24,28532.5655,113.872598,-67.610099,1249.84,1029.35,-768.33,481.51,-58.69,-33.18,-9999.0
1,2017,24,28532.8212,113.873047,-67.610024,1244.53,999.59,-733.12,511.41,-57.32,-34.69,-9999.0
2,2017,24,28533.0771,113.873496,-67.609949,1204.04,1002.24,-695.16,508.88,-59.32,-31.16,-9999.0
3,2017,24,28533.3339,113.873947,-67.609874,1205.95,997.76,-692.50,513.45,-55.82,-34.47,-9999.0
4,2017,24,28533.5894,113.874396,-67.609800,1192.47,1001.67,-682.83,509.64,-57.73,-31.19,-9999.0
...,...,...,...,...,...,...,...,...,...,...,...,...
6403,2017,24,30173.8704,116.694121,-67.093093,1248.37,1381.87,-1133.50,114.87,NaN,-12.38,-9999.0
6404,2017,24,30174.1262,116.694556,-67.093014,1247.99,1381.88,-1133.19,114.80,NaN,-12.80,-9999.0
6405,2017,24,30174.3827,116.694992,-67.092935,1249.91,1381.95,-1135.22,114.69,-77.14,-11.87,-9999.0
6406,2017,24,30174.6387,116.695427,-67.092856,1262.79,1381.78,-1147.98,114.81,-79.21,-12.26,-9999.0


In [18]:
# KCTree worked the best here because there are some minor inconsistencies between the L1B and L2 data (e.g. float32 and float64 coordinates), so we need to find the nearest neighbor in the L2 data for each L1B trace.
# Make dtype consistent for both: radargram coords are originally float32, but L2 coords are float64, so we need to convert them to float32 for the KDTree to work properly.
l1b_lon = radargram_ds["lon"].values.astype(np.float64)
l1b_lat = radargram_ds["lat"].values.astype(np.float64)

l2_lon = l2_df["LON"].values.astype(np.float64)
l2_lat = l2_df["LAT"].values.astype(np.float64)

# stack
l1b_coords = np.column_stack([l1b_lon, l1b_lat])
l2_coords = np.column_stack([l2_lon, l2_lat])

# build tree on l2_df (the side we're looking up / pulling matches from)
tree = cKDTree(l2_coords)

# 1e-5 is 1.11 m longitude
MATCHING_TOLERANCE = 1e-5  # degrees - 1e-2 is likely too loose
distances, indices = tree.query(l1b_coords, distance_upper_bound = MATCHING_TOLERANCE)

# boolean indicator of whether each L1B trace has a match in the L2 data within the matching tolerance
has_match = distances < MATCHING_TOLERANCE

# where has_match is True, use the index from indices, otherwise use 0 to avoid indexing errors
safe_indices = np.where(has_match, indices, 0)  # placeholder for unmatched rows

matched_l2_subset = l2_df.iloc[safe_indices].reset_index(drop = True)

# fill rows that weren't matched with NaN
matched_l2_subset.loc[~ has_match] = np.nan

# Communicate how many matches were found
if has_match.sum() == len(l1b_coords):
    print(f"Matched: {has_match.sum()} / {len(l1b_coords)}")
else:
    # print in red to indicate a partial match
    print(f"\033[91mMatched: {has_match.sum()} / {len(l1b_coords)}\033[0m")

Matched: 4000 / 4000


In [11]:
# speed of light in vacuum
print("Speed of light in vacuum (m/s):", C)

# relative permittivity (epsilon_r) of ice (~ 3.15 is standard for glacial ice)
print("Relative permittivity of ice:", EPSILON_ICE)

# refractive index of ice
n = np.sqrt(EPSILON_ICE)
print("Refractive index of ice:", n)

# speed of light in ice
v_ice = C / n
print("Speed of light in ice (m/s):", v_ice)

Speed of light in vacuum (m/s): 299792458
Relative permittivity of ice: 3.15
Refractive index of ice: 1.7748239349298849
Speed of light in ice (m/s): 168913914.27614674


In [12]:
air_gap_m = radargram_ds.altitude.values - radargram_ds.l2_SURFACE_ELEVATION.values  # plane altitude above surface
print(air_gap_m)
# print(radargram_ds.l2_SRF_RNG.values)

AttributeError: 'Dataset' object has no attribute 'l2_SURFACE_ELEVATION'